In [ ]:
!pip install mlflow optuna imbalanced-learn lightgbm

In [ ]:
!pip install dagshub

In [ ]:
!pip install dagshub mlflow

In [ ]:
import dagshub
import mlflow

dagshub.init(repo_owner="shreychauhan02", repo_name="my-mlflow-project", mlflow=True)

mlflow.set_tracking_uri("https://dagshub.com/shreychauhan02/my-mlflow-project.mlflow")
mlflow.set_experiment('LightGBM-HyP-Tuning-v2')

Accessing as shreychauhan02

Initialized MLflow to track repo "shreychauhan02/my-mlflow-project"

Repository shreychauhan02/my-mlflow-project initialized!

<Experiment: artifact_location='mlflow-artifacts:/b2743c77aaf14165a9ce9e1b3e428964', creation_time=1790243195898, effective_trace_archival_retention=None, experiment_id='7', last_update_time=1790243195898, lifecycle_stage='active', name='LightGBM-HyP-Tuning-v2', tags={}, trace_location=None, workspace='default'>

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report
from imblearn.over_sampling import SMOTE
import mlflow,mlflow.sklearn,optuna
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import optuna.visualization


In [ ]:
import pandas as pd
import numpy as np

df=pd.read_csv("../dataset/pre.csv")
df.head()

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [ ]:

# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})


In [ ]:

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])
df['clean_comment'] = df['clean_comment'].fillna('')


In [ ]:
# Step 3: Split raw text FIRST, then vectorize (fit on train only)
ngram_range = (1, 3)   # Trigram
max_features = 1000

X_text = df['clean_comment']
y = df['category']

X_train_text, X_temp, y_train, y_temp = train_test_split(X_text, y, test_size=0.25, random_state=42, stratify=y)
X_val_text, X_test_text, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.4, random_state=42, stratify=y_temp)

vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train = vectorizer.fit_transform(X_train_text)  # vocab learned from train only
X_val = vectorizer.transform(X_val_text)
X_test = vectorizer.transform(X_test_text)

In [ ]:
# Step 4: Apply SMOTE to the TRAINING split only (never touch val/test)
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

In [ ]:
# Step 5: train/val/test splits + vectorization + resampling all done above,
# in the leak-free order (split -> fit vectorizer on train -> resample train).

In [ ]:
X_train_dense = X_train
X_val_dense = X_val
X_test_dense = X_test

In [ ]:


# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test, params, trial_number):
    with mlflow.start_run():
        # Log model type and trial number
        mlflow.set_tag("mlflow.runName", f"Trial_{trial_number}_{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Log hyperparameters
        for key, value in params.items():
            mlflow.log_param(key, value)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(
                  model,
                  f"{model_name}_model",
                  skops_trusted_types=[
                      "collections.OrderedDict",
                      "lightgbm.basic.Booster",
                      "lightgbm.sklearn.LGBMClassifier",
                  ],
              )
        return accuracy


In [ ]:
# !pip uninstall -y lightgbm
# !pip install lightgbm --install-option=--gpu --install-option="--opencl-include-dir=/usr/local/cuda/include/" --install-option="--opencl-library=/usr/local/cuda/lib64/libOpenCL.so"

In [ ]:
def objective_lightgbm(trial):
    n_estimators = trial.suggest_int('n_estimators', 100, 1000)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 15)
    num_leaves = trial.suggest_int('num_leaves', 20, 150)
    min_child_samples = trial.suggest_int('min_child_samples', 10, 100)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    reg_alpha = trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True)
    reg_lambda = trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True)

    params = {
        'n_estimators': n_estimators,
        'learning_rate': learning_rate,
        'max_depth': max_depth,
        'num_leaves': num_leaves,
        'min_child_samples': min_child_samples,
        'colsample_bytree': colsample_bytree,
        'subsample': subsample,
        'reg_alpha': reg_alpha,
        'reg_lambda': reg_lambda
    }

    model = LGBMClassifier(**params,
                           random_state=42,
                           device='gpu',
                           gpu_platform_id=0,
                           gpu_device_id=0)

    # Tune on the VALIDATION set only; the test set is used once at the end.
    model.fit(X_train_dense, y_train)
    return accuracy_score(y_val, model.predict(X_val_dense))

In [ ]:
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=100)

    best_params = study.best_params
    best_model = LGBMClassifier(n_estimators=best_params['n_estimators'],
                                 learning_rate=best_params['learning_rate'],
                                 max_depth=best_params['max_depth'],
                                 num_leaves=best_params['num_leaves'],
                                 min_child_samples=best_params['min_child_samples'],
                                 colsample_bytree=best_params['colsample_bytree'],
                                 subsample=best_params['subsample'],
                                 reg_alpha=best_params['reg_alpha'],
                                 reg_lambda=best_params['reg_lambda'],
                                 random_state=42,
                                 device='gpu',
                                 gpu_platform_id=0,
                                 gpu_device_id=0)

    log_mlflow("LightGBM", best_model, X_train_dense, X_test_dense, y_train, y_test, best_params, "Best")

    optuna.visualization.plot_param_importances(study).show()
    optuna.visualization.plot_optimization_history(study).show()

In [ ]:

# Run the experiment for LightGBM
run_optuna_experiment()

[I 2026-09-24 09:54:38,015] A new study created in memory with name: no-name-d54d317b-1a18-4317-a638-59fa4744ab55


[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 98943
[LightGBM] [Info] Number of data points in the train set: 37850, number of used features: 966
[LightGBM] [Info] Using requested OpenCL platform 0 device 0
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 16 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Warning] GPU acceleration is disabled because no non-trivial dense features can be found
[LightGBM] [Info] Start training from score -1.098586
[LightGBM] [Info] Start training from score -1.098586
[LightGBM] [Info] Start training from score -1.098665
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

2026/09/24 09:55:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Trial_0_LightGBM_SMOTE_TFIDF_Trigrams at: https://dagshub.com/shreychauhan02/my-mlflow-project.mlflow/#/experiments/7/runs/d8004b743f7d4c8da68870cd1111fa24
🧪 View experiment at: https://dagshub.com/shreychauhan02/my-mlflow-project.mlflow/#/experiments/7


[I 2026-09-24 09:55:43,880] Trial 0 finished with value: 0.671034555637747 and parameters: {'n_estimators': 339, 'learning_rate': 0.0013550969361696464, 'max_depth': 8, 'num_leaves': 122, 'min_child_samples': 43, 'colsample_bytree': 0.5084542488418626, 'subsample': 0.6652927651013145, 'reg_alpha': 0.06644589951314836, 'reg_lambda': 0.0011104916447986014}. Best is trial 0 with value: 0.671034555637747.


...

🏃 View run Trial_9_LightGBM_SMOTE_TFIDF_Trigrams at: https://dagshub.com/shreychauhan02/my-mlflow-project.mlflow/#/experiments/7/runs/fcef53afbc064f82a541326f9fa940b1
🧪 View experiment at: https://dagshub.com/shreychauhan02/my-mlflow-project.mlflow/#/experiments/7
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 99017
[LightGBM] [Info] Number of data points in the train set: 37850, number of used features: 975
[LightGBM] [Info] Using requested OpenCL platform 0 device 0
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 16 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Warning] GPU acceleration is disabled because no non-trivial dense features can be found
[LightGBM] [Info] Start training from score -1.098586
[LightGBM] [Info] Start training from score -1.098586
[LightGBM] [Info] Start training from score -1.098665


2026/09/24 10:13:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
[I 2026-09-24 10:13:43,306] Trial 10 finished with value: 0.7144668709711508 and parameters: {'n_estimators': 250, 'learning_rate': 0.006322768849094127, 'max_depth': 14, 'num_leaves': 109, 'min_child_samples': 18, 'colsample_bytree': 0.9888386287676467, 'subsample': 0.9701665452999032, 'reg_alpha': 0.0015319396007226573, 'reg_lambda': 0.0077768619658988925}. Best is trial 4 with value: 0.8033393215682131.


🏃 View run Trial_10_LightGBM_SMOTE_TFIDF_Trigrams at: https://dagshub.com/shreychauhan02/my-mlflow-project.mlflow/#/experiments/7/runs/8d5e167adfef46a0a9c11b265ea19442
🧪 View experiment at: https://dagshub.com/shreychauhan02/my-mlflow-project.mlflow/#/experiments/7
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 98982
[LightGBM] [Info] Number of data points in the train set: 37850, number of used features: 970
[LightGBM] [Info] Using requested OpenCL platform 0 device 0
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 16 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Warning] GPU acceleration is disabled because no non-trivial dense features can be found
[LightGBM] [Info] Start training from score -1.098586
[LightGBM] [Info] Start training from score -1.098586
[LightGBM] [Info] Start training from score -1.098665
[LightGBM] [Warning] No further splits with positive gain, 

2026/09/24 10:15:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
[I 2026-09-24 10:15:23,237] Trial 11 finished with value: 0.7675155870231428 and parameters: {'n_estimators': 499, 'learning_rate': 0.013718858801856339, 'max_depth': 9, 'num_leaves': 101, 'min_child_samples': 20, 'colsample_bytree': 0.6855376622405325, 'subsample': 0.8690830679241519, 'reg_alpha': 0.003814223080727279, 'reg_lambda': 0.03763242632951883}. Best is trial 4 with value: 0.8033393215682131.


🏃 View run Trial_11_LightGBM_SMOTE_TFIDF_Trigrams at: https://dagshub.com/shreychauhan02/my-mlflow-project.mlflow/#/experiments/7/runs/6618e0f666d24bf7b0d25d28ce2f509e
🧪 View experiment at: https://dagshub.com/shreychauhan02/my-mlflow-project.mlflow/#/experiments/7
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 98764
[LightGBM] [Info] Number of data points in the train set: 37850, number of used features: 957
[LightGBM] [Info] Using requested OpenCL platform 0 device 0
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 16 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Warning] GPU acceleration is disabled because no non-trivial dense features can be found
[LightGBM] [Info] Start training from score -1.098586
[LightGBM] [Info] Start training from score -1.098586
[LightGBM] [Info] Start training from score -1.098665
[LightGBM] [Warning] No further splits with positive gain, 

In [ ]:
!nividia-smi

In [ ]:
import lightgbm as lgb
print(lgb.__version__)

# Check if GPU build hai
try:
    test_data = lgb.Dataset([[1,2],[3,4]], label=[0,1])
    params = {'device': 'gpu', 'objective': 'binary'}
    lgb.train(params, test_data, num_boost_round=1)
    print("GPU LightGBM working ✅")
except Exception as e:
    print("GPU NOT working ❌:", e)